# LeetCode #60: Permutation Sequence

https://leetcode.com/problems/permutation-sequence/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n! \cdot n)$ | $O(n)$ |
| **Optimal: Factorial Number System ★** | $O(n^2)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Generate all $n!$ permutations of [1..n] in lexicographic order (e.g. via next_permutation), then return the $k$-th one. For $n = 13$ this means iterating up to $6{,}227{,}020{,}800$ permutations — completely infeasible.

### Optimal: Factorial Number System ★
Convert $k$ to the factorial number system. At each position, divide $k-1$ by $(n-1)!$ to find which remaining digit goes next — the quotient is the index into the remaining digits list. Remove that digit, decrement $n$, and repeat. This uniquely identifies the $k$-th permutation in exactly $n$ steps with no generation of intermediate permutations.

**Constraints:**
* 1 <= n <= 9
* 1 <= k <= n!

## Solutions
### C#

In [ ]:
// Factorial Number System: decode k directly into the k-th permutation
public class Solution {
    public string GetPermutation(int n, int k) {
        // Pre-compute factorials: fact[i] = i!
        int[] fact = new int[n];
        fact[0] = 1;
        for (int i = 1; i < n; i++) fact[i] = fact[i - 1] * i;

        // Working set of available digits in sorted order
        var digits = new List<int>();
        for (int i = 1; i <= n; i++) digits.Add(i);

        k--; // Convert to 0-indexed for clean division
        var sb = new System.Text.StringBuilder();

        for (int i = n - 1; i >= 0; i--) {
            // Which digit from the remaining list occupies this position?
            int idx = k / fact[i];
            sb.Append(digits[idx]);
            // Remove that digit so it cannot be reused
            digits.RemoveAt(idx);
            k %= fact[i];
        }

        return sb.ToString();
    }
}

### Python

In [ ]:
# Factorial Number System: decode k directly into the k-th permutation
import math

class Solution:
    def getPermutation(self, n: int, k: int) -> str:
        # Available digits in sorted order
        digits = list(range(1, n + 1))
        k -= 1  # Convert to 0-indexed for clean division
        result = []

        for i in range(n - 1, -1, -1):
            # How many permutations does each choice at this position lead?
            block = math.factorial(i)
            idx = k // block
            result.append(str(digits[idx]))
            # Consume this digit — it can no longer appear in remaining positions
            digits.pop(idx)
            k %= block

        return ''.join(result)

### Go

In [ ]:
// Factorial Number System: decode k directly into the k-th permutation
package main

import "strconv"

func getPermutation(n int, k int) string {
    // Pre-compute factorials up to (n-1)!
    fact := make([]int, n)
    fact[0] = 1
    for i := 1; i < n; i++ {
        fact[i] = fact[i-1] * i
    }

    // Remaining digits in sorted order
    digits := make([]int, n)
    for i := range digits {
        digits[i] = i + 1
    }

    k-- // 0-index so quotient directly gives the digit position
    result := make([]byte, 0, n)

    for i := n - 1; i >= 0; i-- {
        // Quotient selects which remaining digit belongs at this position
        idx := k / fact[i]
        result = append(result, []byte(strconv.Itoa(digits[idx]))[0])
        // Remove the chosen digit from the pool
        digits = append(digits[:idx], digits[idx+1:]...)
        k %= fact[i]
    }

    return string(result)
}

### Rust

In [ ]:
// Factorial Number System: decode k directly into the k-th permutation
impl Solution {
    pub fn get_permutation(n: i32, k: i32) -> String {
        let n = n as usize;
        // Pre-compute factorials: fact[i] = i!
        let mut fact = vec![1usize; n];
        for i in 1..n {
            fact[i] = fact[i - 1] * i;
        }

        // Mutable pool of remaining digits
        let mut digits: Vec<u8> = (1..=(n as u8)).collect();
        let mut k = (k - 1) as usize; // Convert to 0-indexed
        let mut result = Vec::with_capacity(n);

        for i in (0..n).rev() {
            // Which remaining digit goes at this position?
            let idx = k / fact[i];
            result.push((b'0' + digits[idx]) as char);
            // Consume that digit from the pool
            digits.remove(idx);
            k %= fact[i];
        }

        result.into_iter().collect()
    }
}

## Example Scenarios

**1. Common Case** — $n = 3, k = 3$

**Input:** `n = 3, k = 3`
Permutations of [1,2,3] in order: 123, 132, 213, 231, 312, 321. k=3 → 0-indexed k=2. Position 0: $k // 2! = 1$ → pick digits[1]=2, remaining [1,3], k=0. Position 1: $k // 1! = 0$ → pick digits[0]=1, remaining [3], k=0. Position 2: pick 3. Result: `"213"`.

**2. Slightly Complex** — $n = 4, k = 9$

**Input:** `n = 4, k = 9`
0-indexed k=8. Position 0: $8 // 3! = 8//6 = 1$ → pick 2, k=2, remaining [1,3,4]. Position 1: $2 // 2! = 1$ → pick 3, k=0, remaining [1,4]. Position 2: $0 // 1! = 0$ → pick 1, remaining [4]. Position 3: pick 4. Result: `"2314"`.

**3. Edge Case: Time Factor** — $k = n!$ (last permutation)

**Input:** `n = 9, k = 362880`
0-indexed k = 362879 = $9! - 1$. At every position, the quotient selects the last remaining digit, producing the reverse-sorted permutation "987654321". The formula reaches this in exactly 9 steps regardless of how large $n!$ is.

**4. Edge Case: Space Factor** — $n = 1$

**Input:** `n = 1, k = 1`
Only one permutation: "1". The factorial array is `[1]`, the loop runs once with $k=0$, picks digits[0]=1 immediately. Space is $O(1)$.

**5. Almost-Impossible but Plausible** — $n = 9, k = 1$

**Input:** `n = 9, k = 1`
Lexicographically smallest permutation "123456789". At every step k remains 0 after subtracting, so idx=0 always picks the smallest remaining digit. The full $9! = 362{,}880$ permutations are skipped with 9 divisions.